In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import librosa
import soundfile as sf

import matplotlib.pyplot as plt
import seaborn as sns

import joblib

In [2]:
PROJECT_DIR = Path.cwd().parent

DATA_DIR = PROJECT_DIR / "data"
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"

metadata_path = PROCESSED_DIR / "metadata.csv"

df = pd.read_csv(metadata_path)

df.head()

,recording_id,file_path,actor_id,emotion_id,emotion_label,intensity,statement_id,repetition,modality,vocal_channel,statement_label,duration,sample_rate,channels,frames,subtype
0,03-01-01-01-01-01-01,Y:\Python\AI ML\Month 2\week 8-NLP\49 — SPEECH...,1,1,Neutral,Normal,1,1,Audio-only,Speech,Kids are talking by the door,3.303292,48000,1,158558,PCM_16
1,03-01-01-01-01-02-01,Y:\Python\AI ML\Month 2\week 8-NLP\49 — SPEECH...,1,1,Neutral,Normal,1,2,Audio-only,Speech,Kids are talking by the door,3.336667,48000,1,160160,PCM_16
2,03-01-01-01-02-01-01,Y:\Python\AI ML\Month 2\week 8-NLP\49 — SPEECH...,1,1,Neutral,Normal,2,1,Audio-only,Speech,Dogs are sitting by the door,3.269917,48000,1,156956,PCM_16
3,03-01-01-01-02-02-01,Y:\Python\AI ML\Month 2\week 8-NLP\49 — SPEECH...,1,1,Neutral,Normal,2,2,Audio-only,Speech,Dogs are sitting by the door,3.169833,48000,1,152152,PCM_16
4,03-01-02-01-01-01-01,Y:\Python\AI ML\Month 2\week 8-NLP\49 — SPEECH...,1,2,Calm,Normal,1,1,Audio-only,Speech,Kids are talking by the door,3.536854,48000,1,169769,PCM_16


# Define audio preprocessing configuration

In [3]:
TARGET_SR = 16000

N_FFT = 2048
HOP_LENGTH = 512

N_MFCC = 13
N_MELS = 128

FMIN = 50
FMAX = 7600

In [4]:
# Our function needs to handle three operations:
# 1. Load the audio without automatically resampling it.
# 2. Convert stereo audio into mono.
# 3. Resample it to our target sample rate of 16 kHz.

def preprocess_audio(file_path, target_sr=TARGET_SR):

    # Load audio while preserving the original sample rate
    audio, original_sr = sf.read(file_path)

    # Convert stereo to mono
    if audio.ndim == 2:
        audio = np.mean(audio, axis=1)

    # Resample if the original sample rate differs
    if original_sr != target_sr:
        audio = librosa.resample(
            y=audio,
            orig_sr=original_sr,
            target_sr=target_sr
        )

    return audio.astype(np.float32), target_sr

In [5]:
sample_path = df.iloc[0]["file_path"]

processed_audio, processed_sr = preprocess_audio(sample_path)

print("Processed sample rate:", processed_sr)
print("Processed shape:", processed_audio.shape)
print("Processed duration:", len(processed_audio) / processed_sr)
print("Data type:", processed_audio.dtype)

Processed sample rate: 16000
Processed shape: (52853,)
Processed duration: 3.3033125
Data type: float32


In [8]:
from IPython.display import display,Audio
display(Audio(processed_audio,rate=processed_sr))

# Time-domain feature extraction

In [9]:
def extract_time_domain_features(audio, sr):

    features = {}

    # Amplitude statistics
    features["mean_amplitude"] = np.mean(audio)
    features["std_amplitude"] = np.std(audio)

    features["max_amplitude"] = np.max(audio)
    features["min_amplitude"] = np.min(audio)

    features["peak_to_peak"] = (
        np.max(audio) - np.min(audio)
    )

    # RMS energy
    rms = librosa.feature.rms(
        y=audio,
        frame_length=N_FFT,
        hop_length=HOP_LENGTH
    )[0]

    features["mean_rms"] = np.mean(rms)
    features["std_rms"] = np.std(rms)

    # Zero Crossing Rate
    zcr = librosa.feature.zero_crossing_rate(
        y=audio,
        frame_length=N_FFT,
        hop_length=HOP_LENGTH
    )[0]

    features["mean_zcr"] = np.mean(zcr)
    features["std_zcr"] = np.std(zcr)

    # Duration
    features["duration"] = len(audio) / sr

    return features

In [10]:
sample_path = df.iloc[0]["file_path"]

audio, sr = preprocess_audio(sample_path)

time_features = extract_time_domain_features(audio, sr)

pd.DataFrame([time_features])

,mean_amplitude,std_amplitude,max_amplitude,min_amplitude,peak_to_peak,mean_rms,std_rms,mean_zcr,std_zcr,duration
0,9.149978e-07,0.003997,0.040111,-0.037105,0.077215,0.002327,0.003232,0.38847,0.259118,3.303313


# Spectral feature extraction

In [11]:
def extract_spectral_features(audio, sr):

    features = {}

    # Spectral Centroid
    centroid = librosa.feature.spectral_centroid(
        y=audio,
        sr=sr,
        n_fft=N_FFT,
        hop_length=HOP_LENGTH
    )[0]

    features["mean_centroid"] = np.mean(centroid)
    features["std_centroid"] = np.std(centroid)

    # Spectral Bandwidth
    bandwidth = librosa.feature.spectral_bandwidth(
        y=audio,
        sr=sr,
        n_fft=N_FFT,
        hop_length=HOP_LENGTH
    )[0]

    features["mean_bandwidth"] = np.mean(bandwidth)
    features["std_bandwidth"] = np.std(bandwidth)

    # Spectral Rolloff
    rolloff = librosa.feature.spectral_rolloff(
        y=audio,
        sr=sr,
        n_fft=N_FFT,
        hop_length=HOP_LENGTH,
        roll_percent=0.85
    )[0]

    features["mean_rolloff"] = np.mean(rolloff)
    features["std_rolloff"] = np.std(rolloff)

    # Spectral Contrast
    contrast = librosa.feature.spectral_contrast(
        y=audio,
        sr=sr,
        n_fft=N_FFT,
        hop_length=HOP_LENGTH
    )

    features["mean_contrast"] = np.mean(contrast)
    features["std_contrast"] = np.std(contrast)

    # Spectral Flatness
    flatness = librosa.feature.spectral_flatness(
        y=audio,
        n_fft=N_FFT,
        hop_length=HOP_LENGTH
    )[0]

    features["mean_flatness"] = np.mean(flatness)
    features["std_flatness"] = np.std(flatness)

    return features

In [12]:
sample_path = df.iloc[0]["file_path"]

audio, sr = preprocess_audio(sample_path)

spectral_features = extract_spectral_features(audio, sr)

pd.DataFrame([spectral_features])

,mean_centroid,std_centroid,mean_bandwidth,std_bandwidth,mean_rolloff,std_rolloff,mean_contrast,std_contrast,mean_flatness,std_flatness
0,2589.795314,1239.049312,1927.217735,510.234817,4741.811899,1946.436076,20.883512,13.606242,0.203758,0.227162


# MFCC Feature Extraction

In [13]:
def extract_mfcc_features(audio, sr):

    features = {}

    mfcc = librosa.feature.mfcc(
        y=audio,
        sr=sr,
        n_mfcc=N_MFCC,
        n_fft=N_FFT,
        hop_length=HOP_LENGTH,
        n_mels=N_MELS,
        fmin=FMIN,
        fmax=FMAX
    )

    # Statistical summaries for each coefficient
    for i in range(N_MFCC):

        features[f"mfcc_{i+1}_mean"] = np.mean(mfcc[i])
        features[f"mfcc_{i+1}_std"] = np.std(mfcc[i])

    return features

In [14]:
sample_path = df.iloc[0]["file_path"]

audio, sr = preprocess_audio(sample_path)

mfcc_features = extract_mfcc_features(audio, sr)

pd.DataFrame([mfcc_features])

,mfcc_1_mean,mfcc_1_std,mfcc_2_mean,mfcc_2_std,mfcc_3_mean,mfcc_3_std,mfcc_4_mean,mfcc_4_std,mfcc_5_mean,mfcc_5_std,...,mfcc_9_mean,mfcc_9_std,mfcc_10_mean,mfcc_10_std,mfcc_11_mean,mfcc_11_std,mfcc_12_mean,mfcc_12_std,mfcc_13_mean,mfcc_13_std
0,-683.535217,199.904175,48.853939,66.771332,3.65909,20.055565,16.258852,27.089588,7.555129,16.355112,...,-2.923199,7.84837,1.519526,8.239791,-0.516605,8.555431,2.710079,8.033251,-1.856134,5.3981


# Pitch and F0 feature extraction

In [15]:
def extract_pitch_features(audio, sr):

    features = {}

    f0, voiced_flag, voiced_prob = librosa.pyin(
        audio,
        fmin=librosa.note_to_hz("C2"),
        fmax=librosa.note_to_hz("C7"),
        sr=sr,
        frame_length=N_FFT,
        hop_length=HOP_LENGTH
    )

    # Retain only voiced-frame pitch estimates
    valid_f0 = f0[voiced_flag]

    if len(valid_f0) > 0:

        features["mean_f0"] = np.mean(valid_f0)
        features["median_f0"] = np.median(valid_f0)
        features["std_f0"] = np.std(valid_f0)

        features["min_f0"] = np.min(valid_f0)
        features["max_f0"] = np.max(valid_f0)

        features["pitch_range"] = (
            np.max(valid_f0) - np.min(valid_f0)
        )

    else:

        features["mean_f0"] = np.nan
        features["median_f0"] = np.nan
        features["std_f0"] = np.nan
        features["min_f0"] = np.nan
        features["max_f0"] = np.nan
        features["pitch_range"] = np.nan

    # Proportion of frames classified as voiced
    features["voiced_ratio"] = np.mean(voiced_flag)

    return features

In [16]:
sample_path = df.iloc[0]["file_path"]

audio, sr = preprocess_audio(sample_path)

pitch_features = extract_pitch_features(audio, sr)

pd.DataFrame([pitch_features])

,mean_f0,median_f0,std_f0,min_f0,max_f0,pitch_range,voiced_ratio
0,98.605211,102.635341,26.655946,65.406391,145.986692,80.5803,0.634615


In [17]:
def extract_all_features(audio, sr):

    features = {}

    # Time-domain features
    time_features = extract_time_domain_features(audio, sr)
    features.update(time_features)

    # Spectral features
    spectral_features = extract_spectral_features(audio, sr)
    features.update(spectral_features)

    # MFCC features
    mfcc_features = extract_mfcc_features(audio, sr)
    features.update(mfcc_features)

    # Pitch and F0 features
    pitch_features = extract_pitch_features(audio, sr)
    features.update(pitch_features)

    return features

In [18]:
sample_path = df.iloc[0]["file_path"]

audio, sr = preprocess_audio(sample_path)

sample_features = extract_all_features(audio, sr)

sample_features_df = pd.DataFrame([sample_features])

sample_features_df

,mean_amplitude,std_amplitude,max_amplitude,min_amplitude,peak_to_peak,mean_rms,std_rms,mean_zcr,std_zcr,duration,...,mfcc_12_std,mfcc_13_mean,mfcc_13_std,mean_f0,median_f0,std_f0,min_f0,max_f0,pitch_range,voiced_ratio
0,9.149978e-07,0.003997,0.040111,-0.037105,0.077215,0.002327,0.003232,0.38847,0.259118,3.303313,...,8.033251,-1.856134,5.3981,98.605211,102.635341,26.655946,65.406391,145.986692,80.5803,0.634615


In [19]:
feature_rows = []

for index, row in df.iterrows():

    # Load and preprocess audio
    audio, sr = preprocess_audio(row["file_path"])

    # Extract all 53 features
    features = extract_all_features(audio, sr)

    # Attach recording metadata
    features.update({
        "recording_id": row["recording_id"],
        "file_path": row["file_path"],
        "actor_id": row["actor_id"],
        "emotion_id": row["emotion_id"],
        "emotion_label": row["emotion_label"],
        "intensity": row["intensity"],
        "statement_id": row["statement_id"],
        "repetition": row["repetition"]
    })

    feature_rows.append(features)

    # Display progress
    if (index + 1) % 100 == 0:
        print(f"Processed {index + 1}/{len(df)} recordings")

# Combine everything into one DataFrame
feature_df = pd.DataFrame(feature_rows)

feature_df.head()

Processed 100/1440 recordings
Processed 200/1440 recordings
Processed 300/1440 recordings
Processed 400/1440 recordings
Processed 500/1440 recordings
Processed 600/1440 recordings
Processed 700/1440 recordings
Processed 800/1440 recordings
Processed 900/1440 recordings
Processed 1000/1440 recordings
Processed 1100/1440 recordings
Processed 1200/1440 recordings
Processed 1300/1440 recordings
Processed 1400/1440 recordings


,mean_amplitude,std_amplitude,max_amplitude,min_amplitude,peak_to_peak,mean_rms,std_rms,mean_zcr,std_zcr,duration,...,pitch_range,voiced_ratio,recording_id,file_path,actor_id,emotion_id,emotion_label,intensity,statement_id,repetition
0,9.149978e-07,0.003997,0.040111,-0.037105,0.077215,0.002327,0.003232,0.388470,0.259118,3.303313,...,80.580300,0.634615,03-01-01-01-01-01-01,Y:\Python\AI ML\Month 2\week 8-NLP\49 — SPEECH...,1,1,Neutral,Normal,1,1
1,7.137581e-07,0.004278,0.047484,-0.041402,0.088887,0.002504,0.003449,0.393392,0.267818,3.336688,...,86.604025,0.780952,03-01-01-01-01-02-01,Y:\Python\AI ML\Month 2\week 8-NLP\49 — SPEECH...,1,1,Neutral,Normal,1,2
2,9.554642e-07,0.005071,0.058643,-0.036928,0.095571,0.002846,0.004172,0.411479,0.278183,3.269938,...,78.903500,0.747573,03-01-01-01-02-01-01,Y:\Python\AI ML\Month 2\week 8-NLP\49 — SPEECH...,1,1,Neutral,Normal,2,1
3,9.176720e-07,0.004866,0.062142,-0.035966,0.098108,0.002660,0.004047,0.400679,0.262333,3.169875,...,73.987775,0.730000,03-01-01-01-02-02-01,Y:\Python\AI ML\Month 2\week 8-NLP\49 — SPEECH...,1,1,Neutral,Normal,2,2
4,8.108808e-07,0.002951,0.025978,-0.025491,0.051469,0.001704,0.002402,0.376575,0.266759,3.536875,...,90.157101,0.666667,03-01-02-01-01-01-01,Y:\Python\AI ML\Month 2\week 8-NLP\49 — SPEECH...,1,2,Calm,Normal,1,1


In [20]:
feature_output_path = PROCESSED_DIR / "extracted_features.csv"

feature_df.to_csv(feature_output_path, index=False)

print("Saved to:", feature_output_path)

Saved to: Y:\Python\AI ML\Month 2\week 8-NLP\49 — SPEECH EMOTION RECOGNITION\data\processed\extracted_features.csv


# Prepare features and target labels

In [21]:
metadata_columns = [
    "recording_id",
    "file_path",
    "actor_id",
    "emotion_id",
    "emotion_label",
    "intensity",
    "statement_id",
    "repetition"
]

X = feature_df.drop(columns=metadata_columns)

y = feature_df["emotion_label"]

groups = feature_df["actor_id"]

In [22]:
from sklearn.model_selection import GroupShuffleSplit

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(X, y, groups=groups)
)

X_train = X.iloc[train_idx].copy()
X_test = X.iloc[test_idx].copy()

y_train = y.iloc[train_idx].copy()
y_test = y.iloc[test_idx].copy()

train_actors = groups.iloc[train_idx]
test_actors = groups.iloc[test_idx]

In [23]:
print("Training emotion distribution:")
print(y_train.value_counts().sort_index())

print("\nTesting emotion distribution:")
print(y_test.value_counts().sort_index())

Training emotion distribution:
emotion_label
Angry        152
Calm         152
Disgust      152
Fearful      152
Happy        152
Neutral       76
Sad          152
Surprised    152
Name: count, dtype: int64

Testing emotion distribution:
emotion_label
Angry        40
Calm         40
Disgust      40
Fearful      40
Happy        40
Neutral      20
Sad          40
Surprised    40
Name: count, dtype: int64


In [25]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

preprocessor = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

X_train_scaled = preprocessor.fit_transform(X_train)

X_test_scaled = preprocessor.transform(X_test)

# Save prepared datasets

In [26]:
MODEL_DIR = PROJECT_DIR / "models"
FEATURE_DIR = PROCESSED_DIR / "features"

FEATURE_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

In [27]:
feature_columns = X.columns.tolist()

X_train_scaled_df = pd.DataFrame(
    X_train_scaled,
    columns=feature_columns,
    index=X_train.index
)

X_test_scaled_df = pd.DataFrame(
    X_test_scaled,
    columns=feature_columns,
    index=X_test.index
)

X_train_scaled_df.to_csv(
    FEATURE_DIR / "X_train.csv",
    index=False
)

X_test_scaled_df.to_csv(
    FEATURE_DIR / "X_test.csv",
    index=False
)

In [28]:
y_train.reset_index(drop=True).to_csv(
    FEATURE_DIR / "y_train.csv",
    index=False
)

y_test.reset_index(drop=True).to_csv(
    FEATURE_DIR / "y_test.csv",
    index=False
)

train_actors.reset_index(drop=True).to_csv(
    FEATURE_DIR / "train_actors.csv",
    index=False
)

test_actors.reset_index(drop=True).to_csv(
    FEATURE_DIR / "test_actors.csv",
    index=False
)

In [29]:
joblib.dump(
    preprocessor,
    MODEL_DIR / "feature_preprocessor.pkl"
)

['Y:\\Python\\AI ML\\Month 2\\week 8-NLP\\49 — SPEECH EMOTION RECOGNITION\\models\\feature_preprocessor.pkl']

In [30]:
joblib.dump(
    feature_columns,
    MODEL_DIR / "feature_columns.pkl"
)

['Y:\\Python\\AI ML\\Month 2\\week 8-NLP\\49 — SPEECH EMOTION RECOGNITION\\models\\feature_columns.pkl']